# 14 · Inference and portable model bundle
**Status: UNEXECUTED FIRST DRAFT.** No trained checkpoint is included by default.

**Purpose:** load an explicitly selected trained AHR checkpoint and export weights, configuration, provenance and the notebook sources required to reconstruct it. **Inputs:** a trusted project checkpoint after completed optimizer updates and all 16 notebook sources. **Outputs:** a local inference bundle containing `.pt`, JSON, Markdown and `.ipynb` files only. It is not a Kaggle submission package.

**Expected runtime:** usually minutes for reading/writing a large checkpoint; actual disk throughput and artifact size are unmeasured. **Checkpoint/resume:** this read-only conversion retains the source checkpoint at `checkpoints/<run_id>/`; each output file is interruption-safe and the manifest is written last. Repeat with the same source identity and destination to finish an interrupted export. A complete manifest is never claimed before hash checks pass.

**Run All:** loads and exports only when `bundle.checkpoint` is configured. Missing trained weights never fall back to an untrained/random agent. Inference preserves the actual Queen–Worker–Scout planner; notebook 15 provides the visible game demo. Sources are notebook-only, including in the bundle.

## Load inference and benchmark definitions

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())
load_exports([13], globals())


## Build a trained model and package reproducible sources

In [ ]:
import shutil
import tempfile


def build_model_from_checkpoint(checkpoint_path, cfg=None, device='cpu'):
    return load_ahr_for_evaluation(checkpoint_path, cfg=cfg, device=device, root=ROOT)


def atomic_torch_save(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f'.tmp-{os.getpid()}')
    try:
        with temporary.open('wb') as stream:
            torch.save(payload, stream)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def atomic_copy_file(source, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + f'.tmp-{os.getpid()}')
    try:
        with Path(source).open('rb') as src, temporary.open('wb') as dst:
            shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
            dst.flush()
            os.fsync(dst.fileno())
        os.replace(temporary, destination)
    finally:
        if temporary.exists():
            temporary.unlink()


def export_bundle(checkpoint_path, destination, cfg=None, root=ROOT):
    root = Path(root).resolve()
    destination = Path(destination)
    if not destination.is_absolute():
        destination = root / destination
    destination = destination.resolve()
    if destination == root or destination == root / 'notebooks':
        raise ValueError('Use a distinct artifacts/<bundle-name> destination.')
    payload, saved_cfg, identity = read_ahr_checkpoint(checkpoint_path, root)
    if identity['step'] < 1:
        raise ValueError('An inference bundle requires trained weights; step-zero recovery fixtures are not trained agents.')
    trained_cfg = copy.deepcopy(payload.get('extra', {}).get('trained_config', saved_cfg))
    training_families = {game.split('-', 1)[0] for game in trained_cfg.get('data', {}).get('train_games', [])}
    training_families.update(payload.get('extra', {}).get('training_game_families', []))
    training_families = sorted(training_families)
    effective_cfg = copy.deepcopy(saved_cfg if cfg is None else cfg)
    if effective_cfg['model'] != saved_cfg['model']:
        raise ValueError('Bundle architecture must match checkpoint dimensions.')
    manifest_path = destination / 'manifest.json'
    if manifest_path.exists():
        prior = json.loads(manifest_path.read_text())
        if prior.get('source_checkpoint', {}).get('sha256') != identity['sha256']:
            raise ValueError('Destination contains a bundle from different weights; choose a new destination.')
    source_hashes = source_manifest(root)
    expected_names = sorted(source_hashes)
    if len(expected_names) != 16:
        raise ValueError('Complete bundle requires exactly the original 16 notebook source files.')
    destination.mkdir(parents=True, exist_ok=True)
    with RunLock(root, 'bundle-' + canonical_hash(str(destination))[:16]):
        # No optimizer is shipped in the inference artifact. The original checkpoint retains resume state.
        weights_path = destination / 'model.pt'
        atomic_torch_save(weights_path, {'model': payload['model'], 'config': effective_cfg,
            'step': identity['step'], 'training_stage': identity['training_stage'], 'source_hashes': source_hashes,
            'extra': {'model_config': effective_cfg, 'trained_config': trained_cfg, 'source_checkpoint': identity,
                      'training_stage': identity['training_stage'], 'training_game_families': training_families}})
        atomic_json(destination / 'config' / 'ahr_v2_xl.json', effective_cfg)
        copied = {}
        for name in expected_names:
            source = root / 'notebooks' / Path(name).name
            if not source.is_file():
                raise FileNotFoundError(f'Notebook missing from source manifest: {name}')
            target = destination / 'notebooks' / source.name
            atomic_copy_file(source, target)
            copied[f'notebooks/{source.name}'] = sha256_file(target)
        readme = """AHR v2 XL inference bundle

This bundle contains trained weights and notebook-only source. It is not an official ARC Prize submission.
Open notebooks/00_setup_and_hardware.ipynb in a Python 3.12 kernel to review pinned dependencies. Avoid rerunning setup against an unrelated project.
From the bundle root, load the tagged exports as documented in the notebooks; build_model_from_checkpoint('model.pt', device='cuda:0') reconstructs the agent.
Open notebook 15 for replay/fresh-game visualization after selecting this bundle's model.pt. No trained checkpoint is silently replaced with random weights.
The source checkpoint retains optimizer/RNG/sampler state for training continuation; model.pt is an inference-only artifact.
Raw public observations and predictions remain separate. Lorentz geometry and low energy do not verify unknown game rules.
"""
        readme_path = destination / 'README.md'
        temporary_readme = destination / 'README.md.tmp'
        temporary_readme.write_text(readme)
        os.replace(temporary_readme, readme_path)
        files = {**copied, 'model.pt': sha256_file(weights_path),
                 'config/ahr_v2_xl.json': sha256_file(destination / 'config' / 'ahr_v2_xl.json'),
                 'README.md': sha256_file(readme_path)}
        manifest = {'schema_version': 1, 'status': 'complete', 'kind': 'inference_bundle',
                    'source_checkpoint': identity, 'training_step': identity['step'], 'model_config': effective_cfg,
                    'trained_config': trained_cfg, 'training_stage': identity['training_stage'],
                    'training_game_families': training_families, 'source_hashes': source_hashes, 'files_sha256': files,
                    'runtime': runtime_fingerprint(), 'benchmark_results': None,
                    'trained_source_hashes': payload.get('source_hashes'),
                    'source_changed_since_checkpoint': payload.get('source_hashes') != source_hashes,
                    'validation': 'file integrity only; inference execution and benchmark are separate stages',
                    'submission_packaging': 'not implemented; official Kaggle runner must be verified separately'}
        for name, digest in files.items():
            if sha256_file(destination / name) != digest:
                raise IOError(f'Bundle integrity failure for {name}')
        atomic_json(manifest_path, manifest)
        return manifest


def verify_bundle(destination):
    directory = Path(destination)
    manifest = json.loads((directory / 'manifest.json').read_text())
    if manifest.get('status') != 'complete':
        raise ValueError('Bundle manifest is incomplete.')
    for name, digest in manifest['files_sha256'].items():
        path = directory / name
        if not path.is_file() or sha256_file(path) != digest:
            raise ValueError(f'Missing or changed bundle artifact: {name}')
    return {'status': 'integrity_verified', 'files': len(manifest['files_sha256']), 'source_checkpoint': manifest['source_checkpoint']}


## Select, export and verify the artifact

In [ ]:
cfg = load_config()
cfg.setdefault('bundle', {})
cfg['bundle'].setdefault('checkpoint', None)
cfg['bundle'].setdefault('destination', 'artifacts/ahr-v2-xl-v1')
checkpoint = cfg['bundle']['checkpoint']
if checkpoint:
    bundle_manifest = export_bundle(checkpoint, cfg['bundle']['destination'], cfg=cfg)
    destination = Path(cfg['bundle']['destination'])
    if not destination.is_absolute():
        destination = ROOT / destination
    print(verify_bundle(destination))
    print('Bundle ready for inference verification in notebook 15:', destination)
else:
    print('BUNDLE NOT CREATED: set bundle.checkpoint to the trained checkpoint selected after notebook 13.')


## Next review
Use notebook 15 to inspect real observations, selected actions, per-role reasoning traces and the difference between imagined and observed states. A successful export only establishes artifact integrity. Training convergence, benchmark performance and a working visual demo must be verified separately.